<a href="https://colab.research.google.com/github/gregblast-bot/AutoMagic_Detection/blob/main/MagicCardDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MAGIC CARD ORGANIZER TRAINING**
We are going to try to train the data on google collab because why not learn a new skill today.

Below is the definition of functions

# Functions

In [ ]:
# ------------------------------------------------
# --- IMPORTS
# ------------------------------------------------
import os
import sys
import requests
import time
import math
import random
import json
from pathlib import Path
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont, ImageFilter  # NEW — added ImageDraw, ImageFilter for synthetic generation
import numpy as np
import cv2
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Javascript, HTML
import matplotlib.pyplot as plt
from google.colab.output import eval_js
import base64

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

# ------------------------------------------------
# --- CONFIG
# ------------------------------------------------
NUM_CARD_IMAGES   = 15000     # Scryfall bulk download target
NUM_OTHER_IMAGES  = 15000    # Synthetic non-card images
IMAGE_SIZE        = 96        # All images will be resized to this size
BATCH_SIZE        = 32        # Number of images to train on at once
NUM_CHANNELS      = 1
DOWNLOAD_THREADS  = 16        # NEW — parallel download workers for card images
SCRYFALL_DELAY    = 0.05      # NEW — delay per request

# ------------------------------------------------
# --- PATHS
# ------------------------------------------------
DATA_DIR = Path("/content/mtg_dataset")
CARD_DIR = DATA_DIR / "card"
NOT_CARD_DIR = DATA_DIR / "not_card"
MODEL_DIR = Path("/content/model_output")

TensorFlow version: 2.19.0


In [ ]:
# ------------------------------------------------
# --- FUNCTIONS
# ------------------------------------------------

# Supports bulk urls
def fetch_scryfall_bulk_urls(num_cards=NUM_CARD_IMAGES):
    """
    Use Scryfall's bulk-data endpoint to get image URLs for up to num_cards.
    This avoids hammering the /cards/random endpoint one-by-one.

    Returns a list of (index, image_url) tuples.
    """
    print("Fetching Scryfall bulk data catalog...")
    bulk_resp = requests.get(
        "https://api.scryfall.com/bulk-data",
        headers={"User-Agent": "MTGCardDetector/1.0"},
        timeout=30,
    )
    bulk_resp.raise_for_status()
    bulk_list = bulk_resp.json()["data"]

    # Find the "default_cards" bulk file (smallest complete set)
    default_bulk = None
    for entry in bulk_list:
        if entry["type"] == "default_cards":
            default_bulk = entry
            break

    if not default_bulk:
        # Fallback to oracle_cards if default not found
        for entry in bulk_list:
            if entry["type"] == "oracle_cards":
                default_bulk = entry
                break

    if not default_bulk:
        raise RuntimeError("Could not find a suitable bulk data source on Scryfall")

    download_uri = default_bulk["download_uri"]
    print(f"Downloading bulk JSON from: {download_uri}")
    print(f"  (This is a large file — may take 1-3 minutes)")

    bulk_data_resp = requests.get(
        download_uri,
        headers={"User-Agent": "MTGCardDetector/1.0"},
        timeout=300,
        stream=True,
    )
    bulk_data_resp.raise_for_status()

    # Parse the JSON — this is the big operation
    print("Parsing bulk JSON...")
    all_cards = json.loads(bulk_data_resp.content)
    print(f"  Total cards in Scryfall database: {len(all_cards)}")

    # Extract image URLs, preferring "small" size for speed/storage
    url_list = []
    for card in all_cards:
        img_url = None

        # Standard cards
        image_uris = card.get("image_uris")
        if image_uris:
            img_url = image_uris.get("small") or image_uris.get("normal")

        # Double-faced / multi-face cards — take front face
        if not img_url:
            faces = card.get("card_faces", [])
            if faces and "image_uris" in faces[0]:
                img_url = faces[0]["image_uris"].get("small") or faces[0]["image_uris"].get("normal")

        if img_url:
            url_list.append(img_url)

    print(f"  Found {len(url_list)} cards with image URLs")

    # Shuffle and trim to requested count
    random.shuffle(url_list)
    url_list = url_list[:num_cards]

    return list(enumerate(url_list))


def _download_single_card(args):
    """Worker function for threaded card download."""
    idx, url = args
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img.save(CARD_DIR / f"card_{idx:06d}.jpg")
        return True
    except Exception:
        return False


def fetch_scryfall_cards(num_cards=NUM_CARD_IMAGES):
    """
    Download MTG card images using bulk data + parallel downloads.
    Much faster than one-by-one random API calls.
    """
    # Step 1: Get all URLs from bulk data
    indexed_urls = fetch_scryfall_bulk_urls(num_cards)
    total = len(indexed_urls)
    print(f"\nDownloading {total} card images with {DOWNLOAD_THREADS} threads...")

    # Step 2: Parallel download
    downloaded = 0
    errors = 0
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=DOWNLOAD_THREADS) as executor:
        futures = {executor.submit(_download_single_card, item): item for item in indexed_urls}

        for future in as_completed(futures):
            if future.result():
                downloaded += 1
            else:
                errors += 1

            done = downloaded + errors
            if done % 1000 == 0:
                elapsed = time.time() - start_time
                rate = done / elapsed if elapsed > 0 else 0
                eta = (total - done) / rate if rate > 0 else 0
                print(f"  Progress: {done}/{total} ({downloaded} ok, {errors} err) "
                      f"— {rate:.0f} img/s, ETA {eta/60:.1f} min")

    elapsed = time.time() - start_time
    print(f"\nDone! Downloaded {downloaded} card images in {elapsed/60:.1f} min "
          f"({errors} errors)")
    return downloaded

def fetch_non_card_images(num_images=NUM_OTHER_IMAGES):
  """
  Download random non-card images for negative samples.
  """

  print(f"Fetching {num_images} non-card images...")
  downloaded = 0
  errors = 0
  max_errors = 50

  while downloaded < num_images and errors < max_errors:
      try:
          # Picsum returns a random image at the requested size
          resp = requests.get(
              f"https://picsum.photos/146/204",  # Similar aspect ratio to MTG cards
              timeout=10,
          )
          resp.raise_for_status()

          img = Image.open(BytesIO(resp.content)).convert("RGB")
          img.save(NOT_CARD_DIR / f"other_{downloaded:04d}.jpg")
          downloaded += 1
          errors = 0

          if downloaded % 50 == 0:
              print(f"  Downloaded {downloaded}/{num_images} non-card images")

          time.sleep(0.3)

      except Exception as e:
          errors += 1
          if errors % 10 == 0:
              print(f"  Warning: {errors} errors so far. Last: {e}")
          time.sleep(0.5)

  print(f"Downloaded {downloaded} non-card images ({errors} final error count)")
  return downloaded

In [ ]:

# ------------------------------------------------
# --- AUGMENTATION PARAMETERS
# ------------------------------------------------
PERSPECTIVE_INTENSITY = 0.3
BRIGHTNESS_MAX_DELTA = 0.3
BLUR_MAX_KERNEL = 7
NOISE_FACTOR = 0.05


# ------------------------------------------------
# --- AUGMENTATION FUNCTIONS
# ------------------------------------------------
def perspective_warp_cv(image_np):
    h, w = image_np.shape[:2]
    max_shift = int(min(h, w) * PERSPECTIVE_INTENSITY)
    if max_shift < 2:
        return image_np

    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])

    def _shift():
        return random.randint(max_shift // 2, max_shift)

    style = random.choice(["top_narrow", "bottom_narrow", "left_narrow", "right_narrow"])

    if style == "top_narrow":
        dst = np.float32([[_shift(), _shift()], [w - _shift(), _shift()], [w, h], [0, h]])
    elif style == "bottom_narrow":
        dst = np.float32([[0, 0], [w, 0], [w - _shift(), h - _shift()], [_shift(), h - _shift()]])
    elif style == "left_narrow":
        dst = np.float32([[_shift(), _shift()], [w, 0], [w, h], [_shift(), h - _shift()]])
    else:
        dst = np.float32([[0, 0], [w - _shift(), _shift()], [w - _shift(), h - _shift()], [0, h]])

    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(image_np, M, (w, h),
                                borderMode=cv2.BORDER_CONSTANT,
                                borderValue=(0, 0, 0))

def adjust_brightness(image_np):
    img_tensor = tf.image.random_brightness(image_np, max_delta=BRIGHTNESS_MAX_DELTA)
    img_tensor = tf.clip_by_value(img_tensor, 0, 255)

    return img_tensor.numpy().astype(np.uint8)

def gaussian_blur(image_np):
    k = random.choice([3, 5, 7])
    k = min(k, BLUR_MAX_KERNEL)

    return cv2.GaussianBlur(image_np, (k, k), 0)

def add_noise(image_np):
    img_float = image_np.astype(np.float32) / 255.0
    noise = tf.random.normal(shape=img_float.shape, mean=0.0, stddev=NOISE_FACTOR)
    noisy = tf.clip_by_value(img_float + noise, 0.0, 1.0)

    return (noisy.numpy() * 255).astype(np.uint8)

def motion_blur(image_np):
    """Simulate hand motion during capture."""
    k_size = random.choice([5, 7, 9])
    direction = random.choice(["horizontal", "vertical", "diagonal"])
    kernel = np.zeros((k_size, k_size), dtype=np.float32)
    if direction == "horizontal":
        kernel[k_size // 2, :] = 1.0
    elif direction == "vertical":
        kernel[:, k_size // 2] = 1.0
    else:
        np.fill_diagonal(kernel, 1.0)
    kernel /= kernel.sum()
    return cv2.filter2D(image_np, -1, kernel)


def partial_crop(image_np):
    """Simulate a card partially in/out of frame."""
    h, w = image_np.shape[:2]
    crop_frac = random.uniform(0.2, 0.5)
    side = random.choice(["top", "bottom", "left", "right"])
    bg_val = random.randint(20, 80)
    result = np.full_like(image_np, bg_val)
    if side == "top":
        cut = int(h * crop_frac)
        result[cut:, :] = image_np[:h - cut, :]
    elif side == "bottom":
        cut = int(h * crop_frac)
        result[:h - cut, :] = image_np[cut:, :]
    elif side == "left":
        cut = int(w * crop_frac)
        result[:, cut:] = image_np[:, :w - cut]
    else:
        cut = int(w * crop_frac)
        result[:, :w - cut] = image_np[:, cut:]
    return result

# ------------------------------------------------
# --- Create the dataset here
# ------------------------------------------------

# Set the operations for the card creation set
operations = [
    perspective_warp_cv,
    adjust_brightness,
    gaussian_blur,
    add_noise,
    motion_blur,
    partial_crop,
]

def generate_augmented_dataset(augments_per_image=1):
    """
    For each image in CARD_DIR, apply a random subset of operations
    and save the augmented results into CARD_DIR alongside originals.
    """
    image_paths = list(CARD_DIR.glob("*.jpg"))
    print(f"Augmenting {len(image_paths)} images, {augments_per_image}x each...")

    count = 0
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        for i in range(augments_per_image):
            augmented = img.copy()

            num_ops = random.randint(1, len(operations))
            chosen_ops = random.sample(operations, num_ops)

            for op in chosen_ops:
                augmented = op(augmented)

            out_name = f"{img_path.stem}_aug{i:02d}.jpg"
            cv2.imwrite(str(CARD_DIR / out_name), augmented)
            count += 1

        # --- NEW: progress logging so you're not staring at silence ---
        if (count // augments_per_image) % 5000 == 0 and count > 0:
            print(f"  Augmented {count // augments_per_image}/{len(image_paths)} originals...")

    print(f"Saved {count} augmented images to {CARD_DIR}")
    return count

In [ ]:
# ------------------------------------------------
# --- Generate instead of download augmentation
# ------------------------------------------------

def _generate_random_gradient(size):
    """Generate a random linear gradient image."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    c1 = np.array([random.randint(0, 255) for _ in range(3)])
    c2 = np.array([random.randint(0, 255) for _ in range(3)])

    direction = random.choice(["horizontal", "vertical", "diagonal"])
    for i in range(size):
        t = i / max(size - 1, 1)
        color = (c1 * (1 - t) + c2 * t).astype(np.uint8)
        if direction == "horizontal":
            img[:, i] = color
        elif direction == "vertical":
            img[i, :] = color
        else:
            for j in range(size):
                td = (i + j) / (2 * max(size - 1, 1))
                img[i, j] = (c1 * (1 - td) + c2 * td).astype(np.uint8)
    return img


def _generate_random_shapes(size):
    """Generate an image with random geometric shapes."""
    bg_color = tuple(random.randint(0, 255) for _ in range(3))
    img = Image.new("RGB", (size, size), bg_color)
    draw = ImageDraw.Draw(img)

    num_shapes = random.randint(3, 15)
    for _ in range(num_shapes):
        color = tuple(random.randint(0, 255) for _ in range(3))
        shape_type = random.choice(["rect", "ellipse", "line", "polygon"])
        x1, y1 = random.randint(0, size - 1), random.randint(0, size - 1)
        x2, y2 = random.randint(0, size - 1), random.randint(0, size - 1)

        # Ensure x1 <= x2 and y1 <= y2 (required by rect/ellipse)
        if x1 > x2: x1, x2 = x2, x1
        if y1 > y2: y1, y2 = y2, y1
        if x1 == x2: x2 = min(x2 + 1, size - 1)
        if y1 == y2: y2 = min(y2 + 1, size - 1)

        if shape_type == "rect":
            draw.rectangle([x1, y1, x2, y2], fill=color)
        elif shape_type == "ellipse":
            draw.ellipse([x1, y1, x2, y2], fill=color)
        elif shape_type == "line":
            width = random.randint(1, 8)
            draw.line([x1, y1, x2, y2], fill=color, width=width)
        else:
            num_points = random.randint(3, 6)
            points = [(random.randint(0, size), random.randint(0, size))
                      for _ in range(num_points)]
            draw.polygon(points, fill=color)

    return np.array(img)


def _generate_noise_image(size):
    """Generate pure random noise."""
    noise_type = random.choice(["uniform", "gaussian", "salt_pepper"])
    if noise_type == "uniform":
        return np.random.randint(0, 256, (size, size, 3), dtype=np.uint8)
    elif noise_type == "gaussian":
        mean = random.randint(64, 192)
        std = random.randint(20, 80)
        img = np.random.normal(mean, std, (size, size, 3))
        return np.clip(img, 0, 255).astype(np.uint8)
    else:
        img = np.zeros((size, size, 3), dtype=np.uint8)
        img[:] = [random.randint(0, 255) for _ in range(3)]
        num_pixels = size * size // 4
        for _ in range(num_pixels):
            x, y = random.randint(0, size - 1), random.randint(0, size - 1)
            img[y, x] = [255, 255, 255] if random.random() > 0.5 else [0, 0, 0]
        return img


def _generate_texture_image(size):
    """Generate a procedural texture (stripes, checkerboard, grid, etc.)."""
    pattern = random.choice(["stripes", "checkerboard", "grid", "circles", "waves"])
    img = np.zeros((size, size, 3), dtype=np.uint8)
    c1 = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)
    c2 = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)
    freq = random.randint(4, 20)

    if pattern == "stripes":
        for i in range(size):
            stripe = (i // max(size // freq, 1)) % 2
            img[i, :] = c1 if stripe == 0 else c2

    elif pattern == "checkerboard":
        block = max(size // freq, 1)
        for i in range(size):
            for j in range(size):
                check = ((i // block) + (j // block)) % 2
                img[i, j] = c1 if check == 0 else c2

    elif pattern == "grid":
        img[:] = c1
        spacing = max(size // freq, 2)
        thickness = random.randint(1, 3)
        for i in range(0, size, spacing):
            img[max(0, i - thickness):i + thickness, :] = c2
            img[:, max(0, i - thickness):i + thickness] = c2

    elif pattern == "circles":
        pil_img = Image.new("RGB", (size, size), tuple(c1))
        draw = ImageDraw.Draw(pil_img)
        cx, cy = size // 2, size // 2
        for r in range(0, size, max(size // freq, 2)):
            ring = r % 2
            color = tuple(c1) if ring == 0 else tuple(c2)
            draw.ellipse([cx - r, cy - r, cx + r, cy + r], outline=color, width=2)
        img = np.array(pil_img)

    elif pattern == "waves":
        for i in range(size):
            for j in range(size):
                val = math.sin(i / max(size / freq, 1) * math.pi * 2) + \
                      math.sin(j / max(size / freq, 1) * math.pi * 2)
                t = (val + 2) / 4
                img[i, j] = (c1 * (1 - t) + c2 * t).astype(np.uint8)

    return img


def _generate_blurred_photo_sim(size):
    """Generate something that looks like a blurry, out-of-focus photo."""
    base = _generate_random_shapes(size)
    pil_img = Image.fromarray(base)
    blur_radius = random.randint(3, 12)
    pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return np.array(pil_img)


def _generate_solid_with_text_sim(size):
    """Generate a solid/gradient background with random scribbles."""
    bg = _generate_random_gradient(size)
    pil_img = Image.fromarray(bg)
    draw = ImageDraw.Draw(pil_img)

    num_lines = random.randint(3, 10)
    for _ in range(num_lines):
        y = random.randint(0, size - 1)
        x_start = random.randint(0, size // 3)
        x_end = random.randint(size // 2, size)
        color = tuple(random.randint(0, 255) for _ in range(3))
        draw.line([(x_start, y), (x_end, y)], fill=color, width=random.randint(1, 3))

    return np.array(pil_img)

def generate_camera_realistic_negatives(num_images):
    """Generate negatives that look like what the camera actually sees."""
    print(f"Generating {num_images} camera-realistic negative images...")
    count = 0
    start = time.time()

    generators = [
        _gen_solid_surface, _gen_wood_grain, _gen_fabric_texture,
        _gen_text_document, _gen_partial_occlusion, _gen_blurry_mess,
        _gen_edge_case_rectangle, _gen_dark_frame, _gen_overexposed_frame,
        _gen_gradient_lighting,
    ]

    while count < num_images:
        gen_func = random.choice(generators)
        try:
            img_np = gen_func(IMAGE_SIZE)
            if img_np is not None:
                Image.fromarray(img_np).save(NOT_CARD_DIR / f"neg_{count:06d}.jpg")
                count += 1
                if count % 2000 == 0:
                    print(f"  Generated {count}/{num_images} ({time.time()-start:.0f}s)")
        except Exception:
            continue
    print(f"Generated {count} camera-realistic negatives in {time.time()-start:.0f}s")
    return count


def _gen_solid_surface(size):
    base_colors = [(180,150,120),(100,70,45),(200,200,200),(240,235,230),(40,40,45),(20,60,120),(30,80,40)]
    color = np.array(random.choice(base_colors), dtype=np.float32)
    img = np.full((size, size, 3), color, dtype=np.float32)
    noise = np.random.normal(0, random.uniform(3, 12), img.shape)
    return np.clip(img + noise, 0, 255).astype(np.uint8)

def _gen_wood_grain(size):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    base_r, base_g, base_b = random.randint(120,180), random.randint(80,130), random.randint(40,80)
    freq, phase = random.uniform(0.05, 0.15), random.uniform(0, 2*math.pi)
    for y in range(size):
        wave = math.sin(y * freq + phase) * 15
        for x in range(size):
            grain = math.sin((x + wave) * 0.3) * 10
            img[y,x] = [np.clip(base_r+grain+random.gauss(0,4),0,255),
                         np.clip(base_g+grain*0.7+random.gauss(0,3),0,255),
                         np.clip(base_b+grain*0.4+random.gauss(0,2),0,255)]
    return img

def _gen_fabric_texture(size):
    base = np.array([random.randint(20,200) for _ in range(3)], dtype=np.float32)
    img = np.full((size, size, 3), base, dtype=np.float32)
    spacing = random.randint(2, 5)
    for y in range(size):
        for x in range(size):
            if (x % spacing == 0) or (y % spacing == 0):
                img[y,x] *= random.uniform(0.85, 0.95)
    return np.clip(img + np.random.normal(0, random.uniform(2,8), img.shape), 0, 255).astype(np.uint8)

def _gen_text_document(size):
    bg = random.randint(230, 255)
    img = Image.new("RGB", (size, size), (bg, bg, bg-5))
    draw = ImageDraw.Draw(img)
    tc = random.randint(10, 80)
    y = random.randint(5, 15)
    while y < size - 5:
        lw = random.randint(size//4, size-10)
        draw.rectangle([random.randint(3,15), y, lw, y+random.randint(1,3)], fill=(tc,tc,tc))
        y += random.randint(4, 10)
    return np.array(img)

def _gen_partial_occlusion(size):
    bg = random.choice([(180,150,120),(200,200,200),(40,40,45)])
    img = np.clip(np.full((size,size,3), bg, dtype=np.int16) + np.random.normal(0,5,(size,size,3)).astype(np.int16), 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    draw = ImageDraw.Draw(pil_img)
    skin = random.choice([(220,185,155),(180,140,105),(140,100,70),(90,60,40)])
    cx, cy = random.randint(size//4, 3*size//4), random.randint(size//4, 3*size//4)
    rx, ry = random.randint(size//4, size//2), random.randint(size//6, size//3)
    draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=skin)
    return np.array(pil_img)

def _gen_blurry_mess(size):
    img = Image.new("RGB", (size, size), tuple(random.randint(40,200) for _ in range(3)))
    draw = ImageDraw.Draw(img)
    for _ in range(random.randint(2, 6)):
        c = tuple(random.randint(0,255) for _ in range(3))
        coords = sorted([random.randint(0,size) for _ in range(2)]), sorted([random.randint(0,size) for _ in range(2)])
        draw.rectangle([coords[0][0], coords[1][0], coords[0][1], coords[1][1]], fill=c)
    return np.array(img.filter(ImageFilter.GaussianBlur(radius=random.randint(8,20))))

def _gen_edge_case_rectangle(size):
    bg = np.full((size,size,3), [random.randint(100,220)]*3, dtype=np.uint8)
    pil_img = Image.fromarray(bg)
    draw = ImageDraw.Draw(pil_img)
    w, h = random.randint(size//3, size*2//3), random.randint(size//4, size*3//4)
    x, y = random.randint(0, size-w), random.randint(0, size-h)
    draw.rectangle([x, y, x+w, y+h], fill=tuple(random.randint(0,255) for _ in range(3)), outline=(0,0,0), width=1)
    return np.array(pil_img)

def _gen_dark_frame(size):
    b = random.randint(5, 40)
    return np.clip(np.full((size,size,3), b, dtype=np.float32) + np.random.normal(0, random.uniform(5,20), (size,size,3)), 0, 255).astype(np.uint8)

def _gen_overexposed_frame(size):
    b = random.randint(210, 250)
    return np.clip(np.full((size,size,3), b, dtype=np.float32) + np.random.normal(0, random.uniform(3,10), (size,size,3)), 0, 255).astype(np.uint8)

def _gen_gradient_lighting(size):
    base = np.array([random.randint(80,200) for _ in range(3)], dtype=np.float32)
    img = np.zeros((size,size,3), dtype=np.float32)
    lx, ly = random.choice([(0,0),(size,0),(0,size),(size,size)])
    max_dist = math.sqrt(2) * size
    for y in range(size):
        for x in range(size):
            falloff = 1.0 - (math.sqrt((x-lx)**2 + (y-ly)**2) / max_dist) * 0.5
            img[y,x] = base * falloff
    return np.clip(img + np.random.normal(0, 4, img.shape), 0, 255).astype(np.uint8)

# All synthetic generators with equal weighting
_SYNTH_GENERATORS = [
    _generate_random_gradient,
    _generate_random_shapes,
    _generate_noise_image,
    _generate_texture_image,
    _generate_blurred_photo_sim,
    _generate_solid_with_text_sim,
]


def generate_non_card_images(num_images=NUM_OTHER_IMAGES):
    """
    Generate synthetic non-card images locally
    These provide diverse negative samples: gradients, shapes, noise,
    textures, blurry scenes, and scribble patterns.
    """
    print(f"Generating {num_images} synthetic non-card images...")
    start_time = time.time()
    log_interval = max(num_images // 100, 100)  # ~100 updates, minimum every 100

    for i in range(num_images):
        generator = random.choice(_SYNTH_GENERATORS)
        gen_size = random.choice([96, 128, 146, 160, 200])
        img_np = generator(gen_size)

        img = Image.fromarray(img_np).resize((146, 204), Image.LANCZOS)
        img.save(NOT_CARD_DIR / f"other_{i:06d}.jpg", quality=85)

        if (i + 1) % log_interval == 0 or (i + 1) == num_images:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            eta = (num_images - i - 1) / rate if rate > 0 else 0
            pct = (i + 1) / num_images
            bar_len = 30
            filled = int(bar_len * pct)
            bar = "█" * filled + "░" * (bar_len - filled)
            print(f"\r  [{bar}] {pct:.0%}  {i + 1}/{num_images} — "
                  f"{rate:.0f} img/s, ETA {eta:.0f}s", end="")

    elapsed = time.time() - start_time
    print(f"\nDone! Generated {num_images} non-card images in {elapsed/60:.1f} min")
    return num_images

# Download / Generator dataset

---



In [ ]:
# Clean out old data
for d in [CARD_DIR, NOT_CARD_DIR]:
    if d.exists():
        shutil.rmtree(d)

# Create directories for storage of the data
CARD_DIR.mkdir(parents=True, exist_ok=True)
NOT_CARD_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

cards_downloaded = fetch_scryfall_cards()
augmented_count = generate_augmented_dataset()
total_card_files = len(list(CARD_DIR.glob("*.jpg")))
print(f"\nTotal card images (originals + augments): {total_card_files}")


#print(f"Generating {total_card_files} non-card images to match...")
#non_cards_generated = generate_non_card_images(num_images=total_card_files)
target_negatives = total_card_files
synthetic_count = generate_camera_realistic_negatives(int(target_negatives * 0.85))
picsum_count = fetch_non_card_images(int(target_negatives * 0.15))
non_cards_generated = len(list(NOT_CARD_DIR.glob("*.jpg")))

print(f"\n{'='*50}")
print(f"  DATASET SUMMARY")
print(f"  Card images:     {total_card_files}")
print(f"  Non-card images: {non_cards_generated}")
print(f"  Balance ratio:   {total_card_files / max(non_cards_generated, 1):.2f}:1.00")
print(f"{'='*50}")

Fetching Scryfall bulk data catalog...
  (This is a large file — may take 1-3 minutes)
Parsing bulk JSON...
  Total cards in Scryfall database: 113773
  Found 113611 cards with image URLs

  Progress: 1000/15000 (1000 ok, 0 err) — 117 img/s, ETA 2.0 min
  Progress: 2000/15000 (2000 ok, 0 err) — 96 img/s, ETA 2.3 min
  Progress: 3000/15000 (3000 ok, 0 err) — 91 img/s, ETA 2.2 min
  Progress: 4000/15000 (4000 ok, 0 err) — 90 img/s, ETA 2.0 min
  Progress: 5000/15000 (5000 ok, 0 err) — 87 img/s, ETA 1.9 min
  Progress: 6000/15000 (6000 ok, 0 err) — 87 img/s, ETA 1.7 min
  Progress: 7000/15000 (7000 ok, 0 err) — 87 img/s, ETA 1.5 min
  Progress: 8000/15000 (8000 ok, 0 err) — 89 img/s, ETA 1.3 min
  Progress: 9000/15000 (9000 ok, 0 err) — 90 img/s, ETA 1.1 min
  Progress: 10000/15000 (10000 ok, 0 err) — 90 img/s, ETA 0.9 min
  Progress: 11000/15000 (11000 ok, 0 err) — 90 img/s, ETA 0.7 min
  Progress: 12000/15000 (12000 ok, 0 err) — 90 img/s, ETA 0.6 min
  Progress: 13000/15000 (13000 ok, 0

#

---

# Camera

In [ ]:
# =============================================================
# TEST
# =============================================================

"""
simple camera function - not going to lie, AI helped me. idk Javascript
to stop the camera - right like the video -> more controls -> pause key
"""

test_js = Javascript('''
  async function testCamera() {
    try {
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      const video = document.createElement('video');
      video.srcObject = stream;
      video.autoplay = true;
      video.style.width = '320px';
      document.body.appendChild(video);
      return "SUCCESS: camera opened";
    } catch (err) {
      return "ERROR: " + err.name + " - " + err.message;
    }
  }
''')

display(test_js)
result = eval_js('testCamera()')
print(result)

In [ ]:
# =============================================================
# Live Camera Inference Uncompressed Keras Model
# =============================================================

import io
from IPython.display import display, update_display
from IPython.display import Image as IPyImage

def preprocess_frame_for_model(img_b64):
    """
    Decode a base64 JPEG frame from the browser camera,
    convert to 96x96 grayscale, normalize to [0,1], and
    return a batch-ready tensor.
    """
    img_bytes = base64.b64decode(img_b64)
    img = Image.open(BytesIO(img_bytes)).convert("L")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    img_np = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(np.expand_dims(img_np, axis=-1), axis=0)


def run_live_camera_inference(model, num_frames=200, fps_target=5):
    """
    Capture frames from the browser webcam and run the uncompressed
    Keras model on each one. Displays the frame + prediction overlay.
    """

    # --- INIT: Open camera, create video + canvas in the DOM ---
    print("Opening camera...")
    resolution = eval_js('''
    (async () => {
        var oldV = document.getElementById("_mtg_vid");
        if (oldV) { if (oldV.srcObject) oldV.srcObject.getTracks().forEach(function(t){t.stop()}); oldV.remove(); }
        var oldC = document.getElementById("_mtg_cvs");
        if (oldC) oldC.remove();

        var video = document.createElement("video");
        video.id = "_mtg_vid";
        video.style.display = "none";
        document.body.appendChild(video);

        var stream = await navigator.mediaDevices.getUserMedia({
            video: { width: 320, height: 240, facingMode: "environment" }
        });
        video.srcObject = stream;
        await video.play();

        var canvas = document.createElement("canvas");
        canvas.id = "_mtg_cvs";
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.style.display = "none";
        document.body.appendChild(canvas);

        return String(video.videoWidth) + "x" + String(video.videoHeight);
    })()
    ''')
    print("Camera opened at " + str(resolution))

    delay = 1.0 / fps_target
    print("Running inference — hold a Magic card in front of the camera.")
    print("Interrupt the cell to stop.\n")

    # Create a placeholder image display that we'll update in-place.
    # This avoids clear_output which would destroy our video/canvas DOM elements.
    placeholder = IPyImage(data=b'\x00', format='jpeg', width=320)
    display_handle = display(placeholder, display_id='mtg_live_frame')

    try:
        for frame_idx in range(num_frames):
            t0 = time.time()

            # --- CAPTURE: Find DOM elements by ID, draw frame, return base64 ---
            b64_frame = eval_js('''
            (async () => {
                var v = document.getElementById("_mtg_vid");
                var c = document.getElementById("_mtg_cvs");
                if (!v || !c) return "";
                var ctx = c.getContext("2d");
                ctx.drawImage(v, 0, 0);
                return c.toDataURL("image/jpeg", 0.8).split(",")[1];
            })()
            ''')

            if not b64_frame:
                continue

            # Preprocess for model
            input_tensor = preprocess_frame_for_model(b64_frame)

            # Run inference on the UNCOMPRESSED model
            prediction = model.predict(input_tensor, verbose=0)[0]
            pred_class = np.argmax(prediction)
            confidence = prediction[pred_class]
            is_card = (pred_class == CLASS_CARD)
            label = "CARD" if is_card else "NOT CARD"

            # Decode the original frame for display
            img_bytes = base64.b64decode(b64_frame)
            frame = Image.open(BytesIO(img_bytes))
            frame_np = np.array(frame)

            # Draw prediction overlay on the frame
            frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
            color = (0, 255, 0) if is_card else (0, 0, 255)
            text = label + " " + f"{confidence:.1%}"
            cv2.putText(frame_bgr, text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            # Frame counter in bottom-left
            cv2.putText(frame_bgr, f"Frame {frame_idx + 1}/{num_frames}", (10, frame_bgr.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

            # Show the 96x96 grayscale model input in top-right corner
            model_input_vis = (input_tensor[0, :, :, 0] * 255).astype(np.uint8)
            model_input_vis = cv2.resize(model_input_vis, (96, 96),
                                          interpolation=cv2.INTER_NEAREST)
            h, w = frame_bgr.shape[:2]
            if h >= 101 and w >= 101:
                frame_bgr[5:101, w - 101:w - 5] = \
                    cv2.cvtColor(model_input_vis, cv2.COLOR_GRAY2BGR)

            # Convert to RGB JPEG bytes for IPython display
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            pil_out = Image.fromarray(frame_rgb)
            buf = io.BytesIO()
            pil_out.save(buf, format='JPEG', quality=85)
            jpeg_bytes = buf.getvalue()

            # Update the display in-place (no clear_output needed!)
            update_display(IPyImage(data=jpeg_bytes, format='jpeg'),
                           display_id='mtg_live_frame')

            # Throttle to target FPS
            elapsed = time.time() - t0
            if elapsed < delay:
                time.sleep(delay - elapsed)

    except KeyboardInterrupt:
        print("\nStopped by user.")
    finally:
        # --- STOP: Clean up camera and DOM elements ---
        try:
            eval_js('''
            (async () => {
                var v = document.getElementById("_mtg_vid");
                if (v && v.srcObject) { v.srcObject.getTracks().forEach(function(t){t.stop()}); v.remove(); }
                var c = document.getElementById("_mtg_cvs");
                if (c) c.remove();
                return "stopped";
            })()
            ''')
        except Exception:
            pass
        print("Camera released.")


# Display

---

In [ ]:
def show_augmented_samples(num_samples=5):
    """
    Show original cards next to their augmented versions.
    """

    originals = [p for p in CARD_DIR.glob("*.jpg") if "_aug" not in p.stem]
    num_samples = min(num_samples, len(originals))
    samples = random.sample(originals, num_samples)

    fig, axes = plt.subplots(num_samples, 2, figsize=(6, 4 * num_samples))
    if num_samples == 1:
        axes = [axes]

    for row, path in enumerate(samples):
        img = cv2.imread(str(path))
        if img is None:
            continue

        # Left: original
        axes[row][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title("Original", fontsize=9)

        # Right: find a matching augmented version
        aug_paths = list(CARD_DIR.glob(f"{path.stem}_aug*.jpg"))
        if aug_paths:
            aug_img = cv2.imread(str(random.choice(aug_paths)))
            axes[row][1].imshow(cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB))
            axes[row][1].set_title("Augmented", fontsize=9)
        else:
            axes[row][1].set_title("No augment found", fontsize=9)

        for ax in axes[row]:
            ax.axis('off')

    plt.suptitle("Before and After Augmentation", fontsize=14)
    plt.tight_layout()
    plt.show()

def show_non_card_samples(num_samples=10):
    """
    Display a grid of downloaded non-card images.
    """

    images = list(NOT_CARD_DIR.glob("*.jpg"))
    num_samples = min(num_samples, len(images))
    samples = random.sample(images, num_samples)

    cols = 5
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = axes.flatten() if num_samples > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < len(samples):
            img = cv2.imread(str(samples[i]))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax.set_title(samples[i].name, fontsize=8)
        ax.axis('off')

    plt.suptitle(f"Non-Card Samples ({len(images)} total)", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def display_simple():
  """
  Display a random card and its augmented version.
  """

  card_images = list(CARD_DIR.glob("*.jpg"))
  img = Image.open(random.choice(card_images))
  display(img)

  img_np = np.array(img.convert("L"))
  img_np = cv2.resize(img_np, (IMAGE_SIZE, IMAGE_SIZE))
  warped = perspective_warp_cv(img_np)

  display(Image.fromarray(warped))

# Main

---

In [ ]:
# simple display of the before and after of the sample
display_simple()
show_augmented_samples()
show_non_card_samples()

# Data Preperation

---

In [ ]:
# Load images from folders
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

# Rescale pixels from [0, 255] to [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# ADD augmentation:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.2),
])
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Define TinyConv Model

---

In [ ]:
# model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1)),

#     # First Convolution: Find edges/corners
#     tf.keras.layers.Conv2D(8, (3, 3), strides=(2, 2), padding='same', activation='relu'),

#     # Using padding='valid' ensures MaxPool V1 compatibility
#     # Since the input to this layer is 48x48 (due to the Conv2D stride),
#     # it will downsample to 24x24.
#     tf.keras.layers.MaxPooling2D((2, 2), padding='valid'),

#     # Second Convolution: Find card shapes
#     tf.keras.layers.Conv2D(16, (3, 3), strides=(2, 2), padding='same', activation='relu'),
#     tf.keras.layers.BatchNormalization(),

#     # Flatten and Classify (0 = Not Card, 1 = Card)
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(1, activation='softmax')
# ])

# model.compile(optimizer='adam',
#               loss='binary_crossentropy',
#               metrics=['accuracy'])

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 1), batch_size=1),
    tf.keras.layers.Conv2D(8, (3, 3), strides=(2, 2), padding='same', activation='relu'),

    tf.keras.layers.MaxPooling2D((2, 2), padding='valid'),
    tf.keras.layers.Conv2D(16, (3, 3), strides=(2, 2), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Raw Camera Detection

---





In [ ]:
# =============================================================
#     Added class_weight computation for balanced training
#     Even though Cell 8 balances the file counts, class_weight
#     ensures the loss function treats both classes equally in
#     case there's any remaining imbalance from download errors.
# =============================================================

# Count actual files per class to compute weights
import pathlib
num_card_files = len(list(CARD_DIR.glob("*.jpg")))
num_noncard_files = len(list(NOT_CARD_DIR.glob("*.jpg")))
total = num_card_files + num_noncard_files

print(f"Training set: {num_card_files} card, {num_noncard_files} non-card")

# Compute class weights (inversely proportional to frequency)
# Classes: 0 = first alphabetical folder (card), 1 = second (not_card)
# image_dataset_from_directory assigns labels alphabetically
class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Class order: {class_names}")

# Weight = total / (2 * count_for_class)
class_weight = {}
counts = {}
for i, name in enumerate(class_names):
    count = len(list((DATA_DIR / name).glob("*.jpg")))
    counts[name] = count
    class_weight[i] = total / (2.0 * max(count, 1))
    print(f"  Class {i} ({name}): {count} images, weight = {class_weight[i]:.3f}")

EPOCHS = 30

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[early_stopping, lr_scheduler],  # NEW
)
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Run it! ---
print("=" * 60)
print("  LIVE CAMERA INFERENCE — UNCOMPRESSED KERAS MODEL")
print("=" * 60)
print(f"  Model: {model.count_params()} parameters (full float32)")
print(f"  Input: {IMAGE_SIZE}x{IMAGE_SIZE} grayscale")
print(f"  Output: sigmoid — P(card)")
print("=" * 60)

run_live_camera_inference(model, num_frames=200, fps_target=5)

# Model Accuracy (before quantization)

---

In [ ]:
# Evaluate the original float model on the validation dataset
loss, accuracy = model.evaluate(val_ds)
print(f"Baseline Float Model Accuracy: {accuracy * 100:.2f}%")

# Define TinyConv Model

---

In [ ]:
# Representative dataset needed for full integer quantization
def representative_data_gen():
  # Take 100 batches from the dataset
  for input_value, _ in train_ds.take(100):
    # Loop through each individual image in the batch
    for i in range(input_value.shape[0]):
      # Expand dims to make it [1, 96, 96, 1]
      single_img = tf.expand_dims(input_value[i], axis=0)
      yield [single_img]

# Export as a standard TFLite model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

# Save the model
with open('mtg_detector.tflite', 'wb') as f:
  f.write(tflite_model_int8)

# Model Accuracy (after quantization)

---

In [51]:
def evaluate_tflite_model(tflite_model_path, dataset):
    # Load the TFLite model and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct_predictions = 0
    total_predictions = 0

    # Iterate through the validation dataset
    for images, labels in dataset:
        for i in range(len(images)):
            # TFLite INT8 models expect data in a specific range and type
            input_data = images[i:i+1].numpy()

            # If the model is quantized to INT8, rescale/cast the input
            if input_details['dtype'] == np.int8:
                scale, zero_point = input_details['quantization']
                input_data = input_data / scale + zero_point
                input_data = input_data.astype(np.int8)

            interpreter.set_tensor(input_details['index'], input_data)
            interpreter.invoke()

            # Get the output and compare to label
            output = interpreter.get_tensor(output_details['index'])
            prediction = np.argmax(output[0])

            if prediction == labels[i].numpy():
                correct_predictions += 1
            total_predictions += 1

    return correct_predictions / total_predictions

# Run the evaluation
quantized_accuracy = evaluate_tflite_model('mtg_detector.tflite', val_ds)
print(f"Quantized TFLite Model Accuracy: {quantized_accuracy * 100:.2f}%")

Quantized TFLite Model Accuracy: 94.48%


# Convert to C Array

---

In [52]:
!apt-get update && apt-get install xxd
!xxd -i mtg_detector.tflite > mtg_model_data.cc

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,937 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,032 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,271 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,249 kB]
Fetched 22.7 MB in 5s (4,9

# Compare Metrics

---

In [53]:
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return size / 1024  # Size in KB

# Assuming you saved the baseline Keras model
model.save("baseline_model.h5")

print("--- Final Comparison ---")
print(f"Baseline Accuracy:  {accuracy * 100:.2f}%")
print(f"Quantized Accuracy: {quantized_accuracy * 100:.2f}%")
print(f"Accuracy Drop:      {(accuracy - quantized_accuracy) * 100:.2f}%")
print("------------------------")
print(f"Baseline Size:      {get_file_size('baseline_model.h5'):.2f} KB")
print(f"Quantized Size:     {get_file_size('mtg_detector.tflite'):.2f} KB")

--- Final Comparison ---
Baseline Accuracy:  94.48%
Quantized Accuracy: 94.48%
Accuracy Drop:      0.00%
------------------------
Baseline Size:      55.77 KB
Quantized Size:     9.45 KB
